### Load Question-Answer Pairs

In [1]:
import json

question_type = "open-ended" # available options: "multiple-choice", "open-ended"
dataset = "ptbr" # available options: "ptbr", "ptpt"

with open(f"data/{dataset}-{question_type}-qa-pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

print(f"Loaded {len(qa_pairs)} question-answer pairs from {dataset} dataset ({question_type})")

Loaded 128 question-answer pairs from ptbr dataset (open-ended)


### Select prompt language

In [2]:
prompt_language = dataset # available options: "en"; "ptbr"; "ptpt"

### Expand references for some ptpt qa pairs

In [3]:
import re
from typing import Dict, List

def expand_references_in_string(
	string: str,
	options: Dict[str, str] | None,
	images: Dict[str, str] | None,
	tables: Dict[str, str] | None,
) -> str:
	options = options or {}
	images = images or {}
	tables = tables or {}

	# Find tokens in parentheses like (image1), (table1), etc.
	tokens = re.findall(r"\(([^)]+)\)", string or "")
	# also extract tokens appearing inside option texts
	if options:
		for opt in options.values():
			tokens.extend(re.findall(r"\(([^)]+)\)", str(opt) or ""))
	# Preserve first-seen order without duplicates
	seen: set[str] = set()
	references: List[str] = []
	for t in tokens:
		if t in seen:
			continue
		if t in images or t in tables:
			seen.add(t)
			references.append(t)

	if not references:
		return string

	if prompt_language == "en":
		parts = [string, "\n\nReferenced content:"]
	elif prompt_language == "ptbr":
		parts = [string, "\n\nConteúdo referenciado:"]
	elif prompt_language == "ptpt":
		parts = [string, "\n\nConteúdo referenciado:"]
	for key in references:
		parts.append(f"[{key}]\n{images.get(key) if key in images else tables.get(key, '')}")
	return "\n".join(parts)

### Add prompt instructions and format options

In [4]:
def build_prompt(question_type: str, question_text: str, options_text: Dict[str, str]):
	if question_type == "multiple-choice":
		# Ensure options appear ordered as A..E (only those present)
		ordered_keys = [k for k in ["A", "B", "C", "D", "E"] if k in options_text]
		options_lines = "\n".join(f"{k}) {options_text[k]}" for k in ordered_keys)

		if prompt_language == "en":
			header = (
				"Solve the following math multiple-choice question. Make sure to put the correct option letter (A,B,C,D or E), and only the correct option letter, inside \\boxed{}.\n\n"
			)
			prompt = f"{header}Question: {question_text}\n\n{options_lines}\n"
		elif prompt_language == "ptbr":
			header = (
				"Resolva a seguinte questão de múltipla escolha de matemática. Certifique-se de colocar a letra da opção correta (A,B,C,D or E), e somente a letra da opção correta, dentro de \\boxed{}. Use Português do Brasil para pensar e responder.\n\n"
			)
			prompt = f"{header}Questão: {question_text}\n\n{options_lines}\n"
		elif prompt_language == "ptpt":
			header = (
				"Resolve a seguinte questão de escolha múltipla de matemática. Certifica-te de colocar a letra da opção correta (A,B,C,D ou E), e apenas a letra da opção correta, dentro de \\boxed{}. Usa Português Europeu para pensar e responder.\n\n"
			)
			prompt = f"{header}Questão: {question_text}\n\n{options_lines}\n"
		question_only = f"{question_text}\n\n{options_lines}\n"
	elif question_type == "open-ended":
		if prompt_language == "en":
			header = (
				"Solve the following math open-ended question. Make sure to put the final answer inside \\boxed{}.\n\n"
			)
			prompt = f"{header}Question: {question_text}\n"
		elif prompt_language == "ptbr":
			header = (
				"Resolva a seguinte questão aberta de matemática. Certifique-se de colocar a resposta final dentro de \\boxed{}. Use Português do Brasil para pensar e responder.\n\n"
			)
			prompt = f"{header}Questão: {question_text}\n"
		elif prompt_language == "ptpt":
			header = (
				"Resolve a seguinte questão de resposta aberta de matemática. Certifica-te de colocar a resposta final dentro de \\boxed{}. Usa Português Europeu para pensar e responder.\n\n"
			)
			prompt = f"{header}Questão: {question_text}\n"
		question_only = f"{question_text}\n"
	return question_only, prompt

### Loop that creates a prompt for every item in the input .json file

In [5]:
prompts = []

for item in qa_pairs:
    if question_type == "multiple-choice":
        question = item.get("question_verbatim")
        options = item.get("options_verbatim")
        if not question or not options:
            print(f"Skipping item without valid question/options: {item.get('id', 'unknown id')}")
            continue
        expanded_question = expand_references_in_string(
                question,
                options,
                item.get("images"),
                item.get("tables"),
        )
    elif question_type == "open-ended":
        question = item.get("question_verbatim")
        options = {}
        if not question:
            print(f"Skipping item without valid open-ended question: {item.get('id', 'unknown id')}")
            continue
        expanded_question = expand_references_in_string(
                question,
                options,
                item.get("images"),
                item.get("tables"),
        )
    question_only, prompt_text = build_prompt(question_type, expanded_question, options)
    prompts.append({
        "id": item.get("id"),
        "question": question_only,
        "correct_answer": item.get("correct_option") if question_type == "multiple-choice" else expand_references_in_string(item.get("answer_verbatim"), {}, item.get("images"), item.get("tables")),
        "level": item.get("level"),
        "contains_latex_figure_in_question": item.get("contains_latex_figure_in_question") if dataset == "ptpt" else False,
        "prompt": prompt_text,
    })

### Print prompts

In [6]:
for entry in prompts:
    # Pretty-print with id context
    pid = entry.get("id", "unknown")
    print(f"id={pid}\n{entry.get('prompt','')}\n")

id=1771
Resolva a seguinte questão aberta de matemática. Certifique-se de colocar a resposta final dentro de \boxed{}. Use Português do Brasil para pensar e responder.

Questão: Um retângulo, o qual não é um quadrado, tem lados com comprimentos inteiros, medidos em centímetros. Se o seu perímetro é $n$ centímetros e sua área é $n$ centímetros quadrados, determine $n$.


id=1772
Resolva a seguinte questão aberta de matemática. Certifique-se de colocar a resposta final dentro de \boxed{}. Use Português do Brasil para pensar e responder.

Questão: Observe que $$\frac{1}{n(n+1)} = \frac{1}{n} - \frac{1}{n+1}$$ Assim, podemos calcular a série $$\sum_{n=0}^{\infty} \frac{1}{n(n+1)} = \frac{1}{1 \cdot 2} + \frac{1}{2 \cdot 3} + \frac{1}{3 \cdot 4} + \dots = \left(1 - \frac{1}{2}\right) + \left(\frac{1}{2} - \frac{1}{3}\right) + \left(\frac{1}{3} - \frac{1}{4}\right) + \dots = 1.$$ Sabendo que $$\sum_{n=1}^{\infty} \frac{1}{n^2} = 1 + \frac{1}{2^2} + \frac{1}{3^2} + \dots = \frac{\pi^2}{6},$$ 

### Store created prompts in a .json file

In [7]:
with open(f"prompts/{dataset}-{question_type}-prompts-prompt-language-{prompt_language}.json", "w", encoding="utf-8") as f:
    json.dump(prompts, f, ensure_ascii=False, indent=2)